In [1]:
"""Nero Dual-Arm Core Functions Test"""
from nero.teleop.nero_dual_arm import NeroDualArm
from nero.teleop.nero_teleop_config import NeroDualArmConfig
import logging
import numpy as np
import time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

/home/keyz/miniconda3/envs/agilex_teleop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""Test 1: Configuration"""
config = NeroDualArmConfig(
    # robot_ip="localhost",
    robot_ip='192.168.110.41',
    robot_port=4242,
    use_gripper=True,
    debug=False
)
print("Config created:", config)
print(f"  - robot_ip: {config.robot_ip}")
print(f"  - robot_port: {config.robot_port}")
print(f"  - use_gripper: {config.use_gripper}")
print(f"  - debug: {config.debug}")

Config created: NeroDualArmConfig(id=None, calibration_dir=None, name='nero_dual_arm', robot_ip='192.168.110.41', robot_port=4242, gripper_ip='192.168.110.114', gripper_port=4243, use_gripper=True, gripper_max_open=0.1, gripper_force=2.0, gripper_reverse=False, close_threshold=0.05, control_mode='oculus', debug=False, num_joints_per_arm=7, cameras={}, max_joint_velocity=2.0, max_ee_velocity=0.5, max_joint_delta=0.3)
  - robot_ip: 192.168.110.41
  - robot_port: 4242
  - use_gripper: True
  - debug: False


In [3]:
"""Test 2: Robot Instance Creation"""
robot = NeroDualArm(config)
print(f"Robot instance created: {robot}")
print(f"  - name: {robot.name}")
print(f"  - is_connected: {robot.is_connected}")

Robot instance created: None NeroDualArm
  - name: nero_dual_arm
  - is_connected: False


In [6]:
"""Test 3: Motor Features"""
motor_features = robot.motor_features
print(f"Motor features ({len(motor_features)} keys):")
print(f"  - Left arm joints: {len([k for k in motor_features if 'left_joint' in k])}")
print(f"  - Right arm joints: {len([k for k in motor_features if 'right_joint' in k])}")
print(f"  - Left EE pose: {len([k for k in motor_features if 'left_ee_pose' in k])}")
print(f"  - Right EE pose: {len([k for k in motor_features if 'right_ee_pose' in k])}")

Motor features (14 keys):
  - Left arm joints: 0
  - Right arm joints: 0
  - Left EE pose: 6
  - Right EE pose: 6


In [7]:
"""Test 4: Action Features"""
action_features = robot.action_features
print(f"Action features ({len(action_features)} keys):")
print(f"  - Left joint pos: {len([k for k in action_features if 'left_joint' in k and 'pos' in k])}")
print(f"  - Right joint pos: {len([k for k in action_features if 'right_joint' in k and 'pos' in k])}")
print(f"  - Left delta EE: {len([k for k in action_features if 'left_delta_ee_pose' in k])}")
print(f"  - Right delta EE: {len([k for k in action_features if 'right_delta_ee_pose' in k])}")

Action features (14 keys):
  - Left joint pos: 0
  - Right joint pos: 0
  - Left delta EE: 6
  - Right delta EE: 6


In [8]:
"""Test 5: Connect to Robot"""
robot.connect()
print(f"Connected: {robot.is_connected}")

DeviceAlreadyConnectedError: nero_dual_arm is already connected.

In [5]:
"""Test 6: Get Observation"""
obs = robot.get_observation()
print(f"Observation keys ({len(obs)}):")
print("  Left arm joints:")
for i in range(7):
    key = f'left_joint_{i+1}.pos'
    print(f"    {key}: {obs[key]:.4f}")
print("  Right arm joints:")
for i in range(7):
    key = f'right_joint_{i+1}.pos'
    print(f"    {key}: {obs[key]:.4f}")
print("  Left EE pose:")
for axis in ['x', 'y', 'z', 'rx', 'ry', 'rz']:
    key = f'left_ee_pose.{axis}'
    print(f"    {key}: {obs[key]:.4f}")
print("  Right EE pose:")
for axis in ['x', 'y', 'z', 'rx', 'ry', 'rz']:
    key = f'right_ee_pose.{axis}'
    print(f"    {key}: {obs[key]:.4f}")

INFO:nero.teleop.nero_dual_arm:[TIMING] left robot query: 4.45ms
INFO:nero.teleop.nero_dual_arm:[TIMING] right robot query: 3.19ms
INFO:nero.teleop.nero_dual_arm:[TIMING] camera total: 0.00ms
INFO:nero.teleop.nero_dual_arm:[TIMING] get_observation total: 9.81ms


Observation keys (14):
  Left arm joints:


KeyError: 'left_joint_1.pos'

In [9]:
"""Test 7: Reset Robot"""
robot.reset()
print("Robot reset completed!")
time.sleep(1)
obs_after_reset = robot.get_observation()
print("Observation after reset:")
print(f"  Left EE x: {obs_after_reset['left_ee_pose.x']:.4f}")
print(f"  Right EE x: {obs_after_reset['right_ee_pose.x']:.4f}")

INFO:nero.teleop.nero_dual_arm:[ROBOT] Resetting dual-arm system...


TimeoutExpired: timeout after 30s, when calling remote method robot_go_home

In [ ]:
"""Test 9: Send Cartesian Delta EE Action"""
obs = robot.get_observation()
action = {}
for i in range(7):
    action[f'left_joint_{i+1}.pos'] = 0.0
    action[f'right_joint_{i+1}.pos'] = 0.0
for axis in ['x', 'y', 'z', 'rx', 'ry', 'rz']:
    action[f'left_delta_ee_pose.{axis}'] = 0.0
    action[f'right_delta_ee_pose.{axis}'] = 0.0
action['right_delta_ee_pose.x'] = 0.01
action['left_delta_ee_pose.x'] = 0.01
action['right_gripper_cmd_width'] = 0.5
action['left_gripper_cmd_width'] = 0.5
result = robot.send_action(action)
print("Cartesian delta EE action sent!")


In [ ]:
# YZ 平面正方形轨迹测试
# 正方形边长 0.15m，分为 10 段，每段 0.015m
square_size = 0.15
steps_per_side = 10
delta_per_step = square_size / steps_per_side
num_repeats = 10

print(f"Drawing {square_size*100}cm square with both arms...")
print(f"Steps per side: {steps_per_side}")
print(f"Delta per step: {delta_per_step:.6f}m")
print(f"Number of repeats: {num_repeats}")

# Define square path: right, down, left, up
square_path = [
    {'x': 0, 'y': delta_per_step, 'z': 0},      # Right
    {'x': 0, 'y': 0, 'z': delta_per_step},     # Up
    {'x': 0, 'y': -delta_per_step, 'z': 0},     # Left
    {'x': 0, 'y': 0, 'z': -delta_per_step}       # Down
]

# Repeat the square path num_repeats times
for repeat in range(num_repeats):
    print(f"\nRepeat {repeat + 1}/{num_repeats}")

    for side_idx, direction in enumerate(square_path):
        side_name = ['Right', 'Up', 'Left', 'Down'][side_idx]
        print(f"\nSide {side_idx + 1}: {side_name}")
        
        for step in range(steps_per_side):
            action = {}
            
            # Set joint positions to 0 (no joint control)
            for i in range(7):
                action[f'left_joint_{i+1}.pos'] = 0.0
                action[f'right_joint_{i+1}.pos'] = 0.0
            
            # Set delta EE pose for both arms (same movement)
            for axis in ['x', 'y', 'z', 'rx', 'ry', 'rz']:
                if axis in ['x', 'y', 'z']:
                    action[f'left_delta_ee_pose.{axis}'] = direction.get(axis, 0.0)
                    action[f'right_delta_ee_pose.{axis}'] = direction.get(axis, 0.0)
                else:
                    action[f'left_delta_ee_pose.{axis}'] = 0.0
                    action[f'right_delta_ee_pose.{axis}'] = 0.0
            
            # Send action
            result = robot.send_action(action)
            
            if step % 5 == 0 or step == steps_per_side - 1:
                print(f"  Step {step + 1}/{steps_per_side}: Moving {direction}")
            
            # time.sleep(0.05)


print(f"\nAll {num_repeats} repeats completed!")
obs_final = robot.get_observation()
print(f"Final Left EE position: x={obs_final['left_ee_pose.x']:.6f}, y={obs_final['left_ee_pose.y']:.6f}, z={obs_final['left_ee_pose.z']:.6f}")
print(f"Final Right EE position: x={obs_final['right_ee_pose.x']:.6f}, y={obs_final['right_ee_pose.y']:.6f}, z={obs_final['right_ee_pose.z']:.6f}")

In [ ]:
"""Test 11: Observation Features"""
obs_features = robot.observation_features
print(f"Observation features ({len(obs_features)} keys):")
print(f"  Motor features: {len(robot.motor_features)}")
print(f"  Camera features: {len(robot.cameras_features)}")

In [ ]:
"""Test 12: Camera Features (Empty)"""
cam_features = robot.cameras_features
print(f"Camera features: {cam_features}")

In [ ]:
"""Test 13: Calibration Status"""
print(f"is_calibrated: {robot.is_calibrated()}")

In [ ]:
"""Test 14: Disconnect Robot"""
robot.disconnect()
print(f"Disconnected: {not robot.is_connected}")